### Retiro Fugas

In [4]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-08-01'

filename='actualizacion_ssff_20260817.xlsx'
name_dni='DNI'

ruta_archivo = os.path.join(ruta_diners_ssff, filename)
df = pd.read_excel(ruta_archivo)

df[f"{name_dni}"] = (
    df[f"{name_dni}"]
    .astype(str)
    .str.zfill(8)
)
print(df.shape)
df = df.rename(columns={
    f'{name_dni}': 'NumDoc'
})
df.head()
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)



(13984, 3)


In [2]:
df.shape

(13984, 3)

In [5]:
df.columns

Index(['NumDoc', 'RECENCIA', 'tipo'], dtype='object')

In [6]:
df['fuente'].unique()

array(['Sigma', 'Alpha', 'Beta', 'Gamma', 'Pi', 'Zeta', 'SIGMA'],
      dtype=object)

In [8]:
df["NumDoc"].nunique()


13984

In [9]:
df.shape

(13984, 3)

In [6]:
df.head()

,NumDoc,RECENCIA,tipo
0,00369552,STOCK,ppd
1,00456136,STOCK,ppd
2,00481490,STOCK,ppd
3,00240022,STOCK,ppd
4,00413873,STOCK,ppd


In [7]:

df.to_sql(
    name="cruce_diners_ssff",
    con=engine_kishin,
    if_exists="append",
    index=False,
    chunksize=1000
)


4198

In [10]:
try:
    with engine_kishin.begin() as conn:
        query =f"""            
            UPDATE a
            SET a.RECENCIA = b.RECENCIA
            from DANTALION.dbo.Base_Maestra_Diners a
            inner join DANTALION.dbo.cruce_diners_ssff B
            ON A.NumDoc=B.NumDoc
            WHERE A.fecha_envio>='2026-08-01'
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 13984


In [ ]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE a
            SET a.N_BASE = b.fuente
            from DANTALION.dbo.Base_Maestra_Diners_TC a
            inner join DANTALION.dbo.cruce_dinerstc B
            ON A.NUMERO_DOCUMENTO=B.NUMERO_DOCUMENTO
            WHERE A.fecha_envio>='2026-08-01'
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

In [9]:
server_sql = server_zeus
db_sql = "MAEBA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_MAEBA = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)


In [11]:
df.to_sql(
    name="cruce_dinerstc",
    con=engine_MAEBA,
    if_exists="append",
    index=False,
    chunksize=1000
)


15195

In [14]:
try:
    with engine_MAEBA.begin() as conn:
        query = f"""
            UPDATE a
            SET a.N_BASE = b.fuente
            from MAEBA.ADM_OBJ_TG.tGestionMesDinersTc a
            inner join MAEBA.dbo.cruce_dinerstc B
            ON A.NUMERO_DOCUMENTO COLLATE Latin1_General_CI_AI=B.NUMERO_DOCUMENTO
            WHERE A.AÑO_DURACION_BASE=2026
            and  a.tMesGestion=b.mes
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 50145


In [18]:
from sqlalchemy import text

with engine_kishin.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS cruce_dinerstc"))
    # conn.execute(text("TRUNCATE TABLE tb_funnel_reclutamiento"))

In [11]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Diners", "SP tNumeros diners")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Diners", "SP actualizar diners Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Diners", "SP actualizar diners SA")

SP tNumeros diners | realizado | duración: 5.62 seg
SP actualizar diners Zeus | realizado | duración: 113.87 seg
SP actualizar diners SA | realizado | duración: 2.12 seg
